<a href="https://colab.research.google.com/github/yichungcheng/SSL_MSM_Endoscope_Spine_surgery/blob/main/00_SSLMSM_down_stream.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

InI

In [1]:
                                      # # 1. 設置與安裝函式庫
# # 【關鍵步驟】執行完此代碼後，務必點擊「執行階段」->「重新啟動工作階段」，否則錯誤不會消失！
# !pip uninstall -y numpy torch torchvision tensorflow tf-models-official segmentation_models_pytorch albumentations pycocotools

# # 1. 先安裝 Numpy (通常最新版是 2.x)
# !pip install -q numpy

# # 2. 升級 Scipy 以兼容 Numpy 2.x (解決 _center 導入錯誤)
# !pip install -q -U "scipy>=1.14.0"

# 3. 安裝 PyTorch (會配合 Numpy 2.x)
!pip install -q torch torchvision

# 4. 安裝 TensorFlow (最新版通常兼容 Numpy 2.x)
# 移除 tf-models-official，因为它經常導致 numpy 版本衝突且似乎未被使用
!pip install -q tensorflow

# 5. 安裝其他工具
!pip install -q segmentation_models_pytorch
!pip install -q albumentations
!pip install -q pycocotools

# 2. 匯入必要的函式庫
import os
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pycocotools.coco import COCO
from google.colab import drive
import shutil

# 3. 連接 Google Drive
drive.mount('/content/gdrive')

# 4. 定義路徑
# 您的資料夾名稱
DRIVE_DIR = '/content/gdrive/MyDrive/PMBM/論文程式/MSM'
# DATASET_NAME = 'Endoscope Spine Surgery.v12i.coco'
# UNZIP_PATH = '/content/dataset'


DATA_ROOT = DRIVE_DIR # 假設數據已經在 Drive 資料夾內

MODULE_DIR = os.path.join(DRIVE_DIR, 'ssl_msn')
os.makedirs(MODULE_DIR, exist_ok=True)
MODEL_SAVE_DIR = os.path.join(MODULE_DIR, 'models')
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


# 檢查路徑結構
print(f"數據根目錄: {DATA_ROOT}")

# 檢查 Google Drive 是否連接正常

if os.path.exists('/content/gdrive/'):
    print("Google Drive 連接正常。")
    # 列出目錄內容測試讀取權限
    print(os.listdir('/content/gdrive/')[:5])
else:
    print("Google Drive 未連接或已斷開，請重新執行 drive.mount。")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 12.4 MB/s eta 0:00:00
Mounted at /content/gdrive
數據根目錄: /content/gdrive/MyDrive/PMBM/論文程式/MSM


In [ ]:

if os.path.exists('/content/gdrive/'):
    print("Google Drive 連接正常。")
    # 列出目錄內容測試讀取權限
    print(os.listdir('/content/gdrive/')[:5])
else:
    print("Google Drive 未連接或已斷開，請重新執行 drive.mount。")

data/cholec80_images.py

In [2]:
'''Module for creating TF datasets for Cholec80 dataset'''  # 這個模組用於建立 Cholec80 影像資料集的 TensorFlow Dataset。

import os  # 匯入作業系統路徑工具。
import tensorflow as tf  # 匯入 TensorFlow 主套件。
# import tensorflow_models as tfm  # 移除導致錯誤的套件


_CHOLEC80_PHASES_WEIGHTS = {  # 定義各手術階段類別的權重（通常用於不平衡資料訓練）。
    0: 1.9219914802981897,  # 類別 0 的權重。
    1: 0.19571110990619747,  # 類別 1 的權重。
    2: 0.9849911311229362,  # 類別 2 的權重。
    3: 0.2993075998175712,  # 類別 3 的權重。
    4: 1.942680301399354,  # 類別 4 的權重。
    5: 1.0,  # 類別 5 的權重。
    6: 2.2015858493443123  # 類別 6 的權重。
    }  # 權重字典結束。


_LABEL_NUM_MAPPING = {  # 定義文字標籤到數字類別 ID 的對應。
    'GallbladderPackaging': 0,  # 膽囊包裝階段對應到 0。
    'CleaningCoagulation': 1,  # 清潔凝固階段對應到 1。
    'GallbladderDissection': 2,  # 膽囊剝離階段對應到 2。
    'GallbladderRetraction': 3,  # 膽囊牽引階段對應到 3。
    'Preparation': 4,  # 準備階段對應到 4。
    'ClippingCutting': 5,  # 夾閉與切割階段對應到 5。
    'CalotTriangleDissection': 6  # Calot 三角剝離階段對應到 6。
    }  # 標籤映射字典結束。


_SUBSAMPLE_RATE = 25  # 標註序列取樣間隔（每 25 個標註取 1 個）。


_CHOLEC80_SPLIT = {'train': range(1, 41),  # 訓練集使用 video01 ~ video40。
                   'validation': range(41, 49),  # 驗證集使用 video41 ~ video48。
                   'test': range(49, 81)}  # 測試集使用 video49 ~ video80。


# curr_dir = os.path.dirname(os.path.realpath(__file__))  # 取得目前檔案所在資料夾絕對路徑。
# config_path = os.path.join(curr_dir, 'config.json')  # 組出 config.json 的完整路徑。
config_path = f'{DRIVE_DIR}/cholec80/config.json'
# Revert IMAGE_DIR to the previous value to verify its structure with ls
IMAGE_DIR = f"{DATA_ROOT}/cholec80/cholec80_extracted/frames"


resize = tf.keras.layers.Resizing(224, 224, crop_to_aspect_ratio=False)  # 建立固定縮放到 224x224 的層（不保持長寬比裁切）。


resize_and_center_crop = tf.keras.layers.Resizing(  # 建立縮放到 224x224 並保持比例中心裁切的層。
    224, 224, crop_to_aspect_ratio=True  # 設定輸出尺寸與裁切行為。
)


# 使用 Keras 內建層取代 tensorflow_models 的 RandAugment
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1), # 新增平移
    tf.keras.layers.RandomZoom(0.1), # 新增縮放
    tf.keras.layers.RandomContrast(0.1),
    tf.keras.layers.RandomBrightness(0.1),
])


def randaug(image):  # 定義使用 RandAugment 的影像轉換函式。
  image = resize(image)  # 先將影像縮放到固定尺寸。
  return data_augmentation(image)


def get_train_image_transformation(name):  # 根據名稱回傳訓練時要使用的影像轉換函式。
  if name == 'randaug':  # 若指定 randaug。
    return randaug  # 回傳含 RandAugment 的處理函式。
  else:  # 其他情況。
    return resize  # 回傳只有縮放的處理函式。


class Cholec80ImagesLoader:  # 封裝 Cholec80 影像與標籤讀取流程的類別。
    def __init__(self, data_root, video_ids, batch_size, shuffle=False, augment=resize):  # 初始化資料載入器。
        self.batch_size = batch_size  # 儲存 batch 大小。
        self.video_ids = video_ids  # 儲存要讀取的影片 ID 清單。
        self.data_root = data_root  # 儲存資料根目錄。
        self.batch_size = batch_size  # 再次設定 batch 大小（與上行重複）。
        self.shuffle = shuffle  # 是否在資料集上啟用 shuffle。
        self.augment = augment  # 儲存影像增強/前處理函式。

        self.all_frame_names, self.all_labels  = self.prebuild(video_ids)  # 預先整理所有影格路徑與對應標籤。

    def prebuild(self, video_ids):  # 掃描指定影片，建立影格檔案清單與標籤清單。
        frames_dir = os.path.join(self.data_root, 'frames')  # frames 資料夾路徑。
        annos_dir = os.path.join(self.data_root, 'phase_annotations')  # 相位標註檔資料夾路徑。

        all_labels = []  # 初始化總標籤列表。
        all_frame_names = []  # 初始化總影格路徑列表。
        for video_id in video_ids:  # 逐一處理每部影片。
            video_frames_dir = os.path.join(frames_dir, video_id)  # 當前影片影格資料夾路徑。
            frames = [os.path.join(video_frames_dir, f) for f in os.listdir(video_frames_dir)]  # 收集當前影片所有影格路徑。
            with open(os.path.join(annos_dir, video_id + '-phase.txt'), 'r') as f:  # 開啟當前影片的相位標註文字檔。
                labels = f.readlines()[1:]  # 讀取所有行並跳過標頭行。
            labels = [l.split('\t')[1][:-1] for l in labels]  # 取出每行第二欄位作為文字標籤並移除換行字元。
            labels = [_LABEL_NUM_MAPPING[l] for l in labels[::_SUBSAMPLE_RATE]][:len(frames)]  # 先依取樣率抽樣，再轉成數字標籤，並截到影格數量。
            all_frame_names += frames  # 將當前影片影格路徑加入總列表。
            all_labels += labels  # 將當前影片標籤加入總列表。
        return all_frame_names, all_labels  # 回傳完整影格與標籤列表。

    def parse_image(self, image_path):  # 將影像路徑解析成前處理後的 Tensor。
        img = tf.io.read_file(image_path)  # 讀取影像檔案內容（二進位）。
        img = tf.io.decode_jpeg(img, channels=3)  # 將 JPEG 解碼為 3 通道影像。
        return self.augment(img)  # 套用增強/縮放處理並回傳。

    def parse_label(self, label):  # 標籤解析函式。
        return label  # 目前直接原樣回傳標籤。

    def parse_example(self, image_path, label):  # 將單筆 (影像路徑, 標籤) 轉為模型可用格式。
        return (self.parse_image(image_path),  # 第一個輸出為處理後影像。
                self.parse_label(label))  # 第二個輸出為處理後標籤。

    def parse_example_image_path(self, image_path, label, image_path_):  # 版本二：額外保留原始影像路徑供追蹤。
        return (self.parse_image(image_path),  # 第一個輸出為處理後影像。
                self.parse_label(label),  # 第二個輸出為標籤。
                image_path_)  # 第三個輸出為影像路徑字串。

    def get_tf_dataset(self, with_image_path=False):  # 建立 tf.data.Dataset 管線。
        num_parallel_calls=tf.data.AUTOTUNE  # 使用 AUTOTUNE 自動決定平行處理數。

        ds_frames = tf.data.Dataset.list_files(self.all_frame_names, shuffle=False)  # 由影格路徑清單建立檔案資料集（不打亂）。
        ds_labels = tf.data.Dataset.from_tensor_slices(self.all_labels)  # 由標籤清單建立資料集。
        if with_image_path:  # 若需要在輸出中保留影像路徑。
            ds = tf.data.Dataset.zip((ds_frames, ds_labels, ds_frames))  # 將影像路徑、標籤與影像路徑本身打包。
            ds = ds.map(self.parse_example_image_path, num_parallel_calls=num_parallel_calls)  # 套用含路徑的解析函式。
        else:  # 若不需要輸出影像路徑。
            ds = tf.data.Dataset.zip((ds_frames, ds_labels))  # 僅打包影像路徑與標籤。
            ds = ds.map(self.parse_example, num_parallel_calls=num_parallel_calls)  # 套用一般解析函式。

        ds = ds.batch(self.batch_size)  # 依 batch_size 分批。
        if self.shuffle:  # 若設定要打亂。
            ds = ds.shuffle(1024)  # 以 buffer size 1024 進行隨機打亂。
        ds = ds.prefetch(num_parallel_calls)  # 預先抓取以提升輸送效率。
        return ds  # 回傳建立完成的資料集。


def get_cholec80_images_datasets(data_root, batch_size, train_transformation='randaug', with_image_path=False):  # 建立 train/validation/test 三個 split 的資料集。
    data = {}  # 初始化回傳字典。
    for split, ids_range in _CHOLEC80_SPLIT.items():  # 迭代每個資料切分與其影片編號範圍。
        if split == 'train':  # 若是訓練集。
            ds = Cholec80ImagesLoader(  # 建立 Cholec80ImagesLoader。
                data_root,  # 傳入資料根目錄。
                [f'video{i:02}' for i in ids_range],  # 將範圍轉成 videoXX 格式清單。
                batch_size,  # 傳入 batch 大小。
                augment=get_train_image_transformation(train_transformation)  # 設定訓練使用的影像轉換。
                )
        data[split] = ds.get_tf_dataset(with_image_path)  # 取得 TensorFlow Dataset 並放入對應 split 鍵值。
    return data  # 回傳所有 split 的資料集字典。


if __name__ == '__main__':  # 當此檔案被直接執行時才會進入此區塊。

    # par_dir = os.path.realpath(__file__+ '/../../')  # 推算專案上層資料夾路徑。
    par_dir= DATA_ROOT
    # data_root = os.path.join(par_dir, 'cholec80', 'cholec80')  # 組合預設資料集根目錄。
    data_root = os.path.join(par_dir, 'cholec80', 'cholec80_extracted')  # 組合預設資料集根目錄。
    datasets = get_cholec80_images_datasets(data_root, 8)  # 建立 batch size=8 的資料集。
    for b in datasets['validation']:  # 迭代驗證集第一個 batch。
        print(b[0].shape)  # 印出影像張量形狀。
        print(b[1].shape)  # 印出標籤張量形狀。
        break  # 只看第一個 batch 後跳出迴圈。

(8, 224, 224, 3)
(8,)


utils.py

In [3]:
# 模組說明：提供訓練過程的視覺化、監控與記錄相關工具。
"""Visualization, monitoring and logging utils."""

# 匯入 matplotlib 的 pyplot 介面，方便建立圖表。
import matplotlib.pyplot as plt


# 定義函式：輸入 keras 訓練歷史，繪製 loss 與指定 metrics 的變化圖。
def plot_history(history, mets=['auc_10'], figsize=(15, 3)):
  # 函式說明：繪製 keras.fit 回傳的 history 曲線圖。
  """Plot keras.fit history graphs."""

  # 從 history 中找出所有「訓練 loss」鍵值（包含 loss、但不包含 val）。
  loss_list = [
      # 逐一檢查 history 的鍵值，篩選訓練 loss 名稱。
      s for s in history.history.keys() if 'loss' in s and 'val' not in s
  ]
  # 從 history 中找出所有「驗證 loss」鍵值（包含 loss 且包含 val）。
  val_loss_list = [
      # 逐一檢查 history 的鍵值，篩選驗證 loss 名稱。
      s for s in history.history.keys() if 'loss' in s and 'val' in s
  ]

  # 建立 epoch 序列（從 1 開始），長度以第一個訓練 loss 的紀錄數為準。
  epochs = range(1, len(history.history[loss_list[0]]) + 1)
  # 建立整張圖，大小由 figsize 參數控制。
  plt.figure(figsize=figsize)
  # 建立第一個子圖位置，用來畫 loss 曲線。
  ax = plt.subplot(1, 1 + len(mets), 1)

  # 逐一繪製每個訓練 loss 曲線。
  for l in loss_list:
    # 在目前子圖上繪製藍色訓練 loss，並在圖例顯示最後一個數值。
    ax.plot(
        # x 軸：epochs。
        epochs,
        # y 軸：對應 loss 歷史值。
        history.history[l],
        # 線條樣式：藍色。
        'b',
        # 圖例文字：Train loss + 最後一個 epoch 的 loss 值（格式化到小數三位）。
        label='Train loss ('
        + str(str(format(history.history[l][-1], '.3f')) + ')'),
    )
  # 逐一繪製每個驗證 loss 曲線。
  for l in val_loss_list:
    # 在目前子圖上繪製綠色驗證 loss，並在圖例顯示最後一個數值。
    ax.plot(
        # x 軸：epochs。
        epochs,
        # y 軸：對應驗證 loss 歷史值。
        history.history[l],
        # 線條樣式：綠色。
        'g',
        # 圖例文字：Valid loss + 最後一個 epoch 的 loss 值（格式化到小數三位）。
        label='Valid loss ('
        + str(str(format(history.history[l][-1], '.3f')) + ')'),
    )

  # 設定 loss 子圖標題。
  ax.set_title('Loss')
  # 設定 x 軸標籤。
  ax.set_xlabel('Epochs')
  # 設定 y 軸標籤。
  ax.set_ylabel('Loss')
  # 顯示圖例。
  ax.legend()

  # 逐一處理使用者指定要畫出的 metrics（例如 auc、accuracy、lr）。
  for i, met_name in enumerate(mets):
    # 建立對應 metric 的子圖，位置從第 2 格開始。
    ax = plt.subplot(1, 1 + len(mets), 2 + i)

    # 若 metric 為學習率 lr，直接畫出 lr 曲線（通常沒有 val 對應）。
    if met_name == 'lr':
      # 繪製藍色學習率曲線。
      ax.plot(epochs, history.history['lr'], 'b')

    # 其他 metric（例如 auc、accuracy）則同時畫訓練與驗證曲線。
    else:
      # 找到對應的訓練 metric 名稱（包含 met_name、且不含 val）。
      train_met_name = [
          # 從所有鍵值中篩選訓練 metric。
          s for s in history.history.keys() if met_name in s and 'val' not in s
      ][0]
      # 找到對應的驗證 metric 名稱（包含 met_name、且包含 val）。
      val_met_name = [
          # 從所有鍵值中篩選驗證 metric。
          s for s in history.history.keys() if met_name in s and 'val' in s
      ][0]

      # 繪製訓練 metric 曲線。
      ax.plot(
          # x 軸：epochs。
          epochs,
          # y 軸：訓練 metric 歷史值。
          history.history[train_met_name],
          # 線條樣式：藍色。
          'b',
          # 圖例文字：Train + metric 名稱 + 最後值（小數三位）。
          label=f'Train {met_name} ('
          + str(str(format(history.history[train_met_name][-1], '.3f')) + ')'),
      )
      # 繪製驗證 metric 曲線。
      ax.plot(
          # x 軸：epochs。
          epochs,
          # y 軸：驗證 metric 歷史值。
          history.history[val_met_name],
          # 線條樣式：綠色。
          'g',
          # 圖例文字：Valid + metric 名稱 + 最後值（小數三位）。
          label=f'Valid {met_name} ('
          + str(str(format(history.history[val_met_name][-1], '.3f')) + ')'),
      )

    # 設定 metric 子圖標題（使用 metric 名稱）。
    ax.set_title(met_name)
    # 設定 x 軸標籤。
    ax.set_xlabel('Epochs')
    # 設定 y 軸標籤（使用 metric 名稱）。
    ax.set_ylabel(met_name)
    # 顯示圖例。
    ax.legend()

train_lib.py

In [4]:
"""Helper training functions."""  # 模組說明：提供模型訓練時會重複使用的工具函式。

import functools  # 匯入 functools（目前此檔案未使用，可能預留給後續函式包裝用途）。
import tensorflow as tf  # 匯入 TensorFlow 主套件，供模型、損失函式與 callback 使用。
# from tensorflow_addons import metrics as tfa_metrics  # 匯入 TensorFlow Addons 指標模組並命名為 tfa_metrics。
# from tensorflow_addons import optimizers as tfa_optimizers  # 匯入 TensorFlow Addons 最佳化器模組並命名為 tfa_optimizers。


def get_linear_model(input_dim: int, output_dim: int):  # 定義函式：建立簡單線性模型。
  return tf.keras.Sequential([  # 回傳 Keras Sequential 模型容器。
      tf.keras.layers.Flatten(input_shape=(input_dim,)),  # 第一層：將輸入攤平成一維向量。
      tf.keras.layers.Dense(output_dim),  # 第二層：全連接層，輸出指定維度 logits。
  ])  # 結束模型層列表。


class LinearFineTuneModel(tf.keras.Model):  # 定義自訂模型類別：以 backbone 特徵再接一層投影。
  def __init__(self, backbone, output_dim):  # 建構子：接收主幹模型與輸出維度。
    super().__init__()  # 初始化父類別 tf.keras.Model。
    self.backbone = backbone  # 儲存 backbone 模型供前向傳播使用。
    self.projection = tf.keras.layers.Dense(output_dim)  # 建立輸出投影層（分類頭）。

  def call(self, x):  # 定義前向傳播邏輯。
    return self.projection(self.backbone(x)[1])  # 取 backbone 回傳結果的第 2 個元素並投影成最終輸出。

  def get_config(self):  # 提供模型序列化時的設定取得介面。
    return super().get_config()  # 直接回傳父類別預設設定。


def get_loss(task_type: str):  # 定義函式：依任務型態選擇損失函式。
  if task_type == 'multi_class':  # 若為多類別單標籤分類。
    return tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # 使用稀疏類別交叉熵（輸入為 logits）。
  elif task_type == 'multi_label':  # 若為多標籤分類。
    return tf.keras.losses.BinaryCrossentropy(from_logits=True)  # 使用二元交叉熵（輸入為 logits）。

  else:  # 若任務型態不在支援清單。
    raise ValueError  # 拋出錯誤提醒呼叫端提供正確 task_type。


def get_optimizer(  # 定義函式：依名稱建立最佳化器。
    optimizer_name: str,  # 參數：最佳化器名稱。
    learning_rate: float = 0.001,  # 參數：學習率預設值。
    momentum: float = 0.9,  # 參數：動量預設值。
    weight_decay: float = 0.00001) -> tf.keras.optimizers.Optimizer:  # 參數：權重衰減，並標註回傳型別。
  """Initialize optimizer by its name."""  # 函式說明：透過名稱初始化最佳化器。

  optimizer_name = optimizer_name.lower()  # 將名稱轉小寫，避免大小寫差異造成判斷錯誤。
  if optimizer_name == 'adam':  # 分支：Adam。
    return tf.keras.optimizers.Adam(learning_rate=learning_rate)  # 建立 Adam 並設定學習率。
  elif optimizer_name == 'adamw':  # 分支：AdamW。
    # 使用 Keras 內建的 AdamW
    return tf.keras.optimizers.AdamW(  # 建立 Keras 版本 AdamW。
        learning_rate=learning_rate, weight_decay=weight_decay)  # 設定學習率與權重衰減。
  elif optimizer_name == 'rmsprop':  # 分支：RMSprop。
    return tf.keras.optimizers.RMSprop(  # 建立 RMSprop。
        learning_rate=learning_rate, momentum=momentum)  # 設定學習率與動量。
  elif optimizer_name == 'momentum':  # 分支：帶動量的 SGD。
    return tf.keras.optimizers.SGD(  # 建立 SGD（Momentum 形式）。
        learning_rate=learning_rate,  # 設定學習率。
        momentum=momentum,  # 設定動量。
        weight_decay=weight_decay)  # 設定權重衰減。
  elif optimizer_name == 'sgd':  # 分支：純 SGD。
    return tf.keras.optimizers.SGD(  # 建立 SGD。
        learning_rate=learning_rate, momentum=0, weight_decay=weight_decay)  # 動量固定為 0，並設定其他參數。
  else:  # 若名稱不支援。
    raise ValueError('Optimizer %s not supported' % optimizer_name)  # 拋出包含名稱的錯誤訊息。


class MyF1Score(tf.keras.metrics.Metric): # 改繼承自 tf.keras.metrics.Metric
  def __init__(self, num_classes, average='micro', name='f1_score', **kwargs):
      super().__init__(name=name, **kwargs)
      self.num_classes = num_classes
      self.average = average
      self.f1_score = tf.keras.metrics.F1Score(average=average, name=name)

  def update_state(self, y_true, y_pred, sample_weight=None):
    # y_true 是 sparse (batch,), 轉成 one-hot (batch, num_classes)
    y_true = tf.one_hot(tf.cast(y_true, tf.int32), self.num_classes)
    # 如果 y_true 有多餘維度 (batch, 1, num_classes) 則 squeeze
    if len(y_true.shape) == 3:
        y_true = tf.squeeze(y_true, 1)

    self.f1_score.update_state(y_true, y_pred, sample_weight)

  def result(self):
      return self.f1_score.result()

  def reset_state(self):
      self.f1_score.reset_state()


def get_metrics(task_type: str, num_classes: int):  # 定義函式：依任務型態回傳評估指標清單。
  if task_type == 'multi_class':  # 多類別任務分支。
    return [MyF1Score(num_classes=num_classes,  # 微平均 F1。
                      average='micro',  # 設定平均方式為 micro。
                      name='micro_f1'),  # 指標名稱。
            MyF1Score(num_classes=num_classes,  # 宏平均 F1。
                      average='macro',  # 設定平均方式為 macro。
                      name='macro_f1'),  # 指標名稱。
            tf.keras.metrics.SparseCategoricalAccuracy()]  # 稀疏類別準確率。
  elif task_type == 'multi_label':  # 多標籤任務分支。
    return [tf.keras.metrics.Precision(name='precision'),  # 精確率指標。
            tf.keras.metrics.AUC(curve='PR',  # PR 曲線下面積。
                                 multi_label=True,  # 啟用多標籤模式。
                                 num_labels=num_classes,  # 指定標籤數量。
                                 from_logits=True,  # 指出模型輸出為 logits。
                                 name='pr_auc')]  # 指標名稱。
  else:  # 任務型態不支援分支。
    raise ValueError  # 拋出錯誤提醒。


def get_early_stopping_callback(  # 定義函式：建立 EarlyStopping callback。
    monitor_metric='val_loss',  # 監控指標預設為驗證損失。
    start_from_epoch=20,  # 從第幾個 epoch 後才開始監控停訓。
    patience=5,  # 容忍幾個 epoch 未改善。
    verbose=1,  # 輸出訊息等級。
    mode='auto',  # 自動判斷指標是要最小化或最大化。
    restore_best_weights=True):  # 是否回復最佳權重。

  return tf.keras.callbacks.EarlyStopping(  # 回傳 EarlyStopping 實例。
      monitor=monitor_metric,  # 套用監控指標。
      start_from_epoch=start_from_epoch,  # 套用開始監控 epoch。
      patience=patience,  # 套用 patience。
      verbose=verbose,  # 套用輸出等級。
      mode=mode,  # 套用模式。
      restore_best_weights=restore_best_weights,  # 套用回復最佳權重設定。
  )  # 結束 callback 建立。


def get_checkpoint_callback(  # 定義函式：建立模型檢查點 callback。
    exp_dir,  # 實驗輸出資料夾。
    monitor='val_loss',  # 監控指標。
    verbose=1,  # 輸出訊息等級。
    save_best_only=True,  # 僅儲存最佳模型。
    save_weights_only=False,  # 是否只存權重。
    mode='auto',  # 指標比較模式。
    save_freq='epoch',  # 儲存頻率。
):
  checkpoint_string = '/checkpoints/epoch_{epoch:02d}.keras'  # 定義 checkpoint 檔名模板。
  return tf.keras.callbacks.ModelCheckpoint(  # 回傳 ModelCheckpoint 實例。
      exp_dir + checkpoint_string,  # 設定儲存路徑。
      monitor=monitor,  # 套用監控指標。
      verbose=verbose,  # 套用輸出等級。
      save_best_only=save_best_only,  # 套用是否僅存最佳。
      save_weights_only=save_weights_only,  # 套用是否只存權重。
      mode=mode,  # 套用模式。
      save_freq=save_freq,  # 套用儲存頻率。
  )  # 結束 callback 建立。


def get_tensorboard_callback(exp_dir):  # 定義函式：建立 TensorBoard callback。
  return tf.keras.callbacks.TensorBoard(  # 回傳 TensorBoard 實例。
      log_dir=exp_dir + '/logs',  # 設定 log 輸出資料夾。
      write_graph=False,  # 不寫入計算圖以減少檔案量。
      write_steps_per_second=True,  # 啟用每秒步數統計。
      update_freq='epoch')  # 每個 epoch 更新一次。


def get_reduce_lr_plateau_callback(  # 定義函式：建立學習率下降 callback。
    monitor='val_loss',  # 監控指標。
    factor=0.3,  # 學習率衰減倍率。
    patience=10,  # 幾個 epoch 無改善後觸發。
    verbose=1,  # 輸出訊息等級。
    mode='auto',  # 指標比較模式。
    min_lr=1e-5,  # 學習率下限。
):
  return tf.keras.callbacks.ReduceLROnPlateau(  # 回傳 ReduceLROnPlateau 實例。
      monitor=monitor,  # 套用監控指標。
      factor=factor,  # 套用衰減倍率。
      patience=patience,  # 套用 patience。
      verbose=verbose,  # 套用輸出等級。
      mode=mode,  # 套用模式。
      min_lr=min_lr,  # 套用最小學習率。
  )  # 結束 callback 建立。


def get_learning_rate_step_scheduler_callback(  # 定義函式：建立階梯式學習率排程 callback。
    learning_rate=1e-4,  # 初始學習率參數（供介面一致性使用）。
    factor=0.3,  # 每次命中里程碑時的乘法係數。
    milestones=[30],  # 觸發衰減的 epoch 清單。
    verbose=1,  # 輸出訊息等級。
):
  def scheduler(epoch, learning_rate):  # 內部排程函式：每個 epoch 計算新學習率。
    if epoch in milestones:  # 若當前 epoch 命中里程碑。
      return learning_rate * factor  # 回傳衰減後學習率。
    else:  # 若未命中里程碑。
      return learning_rate  # 保持原學習率不變。

  return tf.keras.callbacks.LearningRateScheduler(  # 回傳 LearningRateScheduler 實例。
      scheduler,  # 指定排程函式。
      verbose=verbose,  # 套用輸出等級。
  )  # 結束 callback 建立。


def get_callbacks(callbacks_names, exp_dir, monitor_metric, learning_rate):  # 定義函式：依名稱集合組裝 callback 清單。
  callbacks = []  # 初始化空清單。
  if 'checkpoint' in callbacks_names:  # 若需要 checkpoint callback。
    callbacks.append(get_checkpoint_callback(exp_dir, monitor_metric))  # 加入 checkpoint callback。
  if 'reduce_lr_plateau' in callbacks_names:  # 若需要 Plateau 降學習率 callback。
    callbacks.append(get_reduce_lr_plateau_callback(monitor_metric))  # 加入 ReduceLROnPlateau callback。
  if 'step_scheduler' in callbacks_names:  # 若需要階梯式學習率排程 callback。
    callbacks.append(get_learning_rate_step_scheduler_callback(  # 加入 LearningRateScheduler callback。
        learning_rate=learning_rate))  # 傳入當前學習率設定。
  if 'early_stopping' in callbacks_names:  # 若需要 EarlyStopping callback。
    callbacks.append(get_early_stopping_callback())  # 加入 EarlyStopping callback。
  if 'tensorboard' in callbacks_names:  # 若需要 TensorBoard callback。
    callbacks.append(get_tensorboard_callback(exp_dir))  # 加入 TensorBoard callback。
  return callbacks  # 回傳組裝完成的 callback 清單。

eval_lib.py

In [5]:
"""Evaluation methods."""  # 模組說明：集中放置模型評估相關函式。

import numpy as np  # 匯入 NumPy，用於陣列操作與統計計算。
from sklearn.metrics import average_precision_score  # 匯入 AP（Average Precision）計算函式。
from sklearn.metrics import classification_report  # 匯入分類報告函式（可回傳 accuracy、precision、recall、f1）。
from sklearn.metrics import precision_recall_fscore_support as score  # 匯入 precision/recall/f1/support 計算函式並命名為 score。
import tensorflow as tf  # 匯入 TensorFlow，用於模型推論與 TensorBoard 紀錄。


def calc_f1(model, ds, agg='video', verbose=0):  # 定義 F1 計算函式；可選擇 frame 或 video 層級聚合。
  video2labels = {}  # 建立字典：記錄每個影片對應的真實標籤序列。
  video2preds = {}  # 建立字典：記錄每個影片對應的預測標籤序列。

  all_labels = []  # 建立清單：累積所有 frame/clip 的真實標籤。
  all_preds = []  # 建立清單：累積所有 frame/clip 的預測標籤。
  for batch in ds:  # 逐批走訪資料集。

    inputs, labels, clip_paths = batch  # 拆出輸入、標籤與 clip 路徑。
    preds = model.predict(inputs, verbose=verbose)  # 以模型對該批輸入做推論。
    preds = tf.argmax(preds, 1)  # 將類別機率/分數轉成最終預測類別索引。

    all_labels += labels.numpy().tolist()  # 將該批真實標籤轉成 Python list 後併入總清單。
    all_preds += preds.numpy().tolist()  # 將該批預測標籤轉成 Python list 後併入總清單。

    if agg == 'video':  # 若要以影片層級評估，需按影片 regroup 各 clip 結果。
      for c, l, p in zip(clip_paths.numpy(), labels.numpy(), preds.numpy()):  # 同步迭代每個 clip 的路徑、標籤與預測。
        c = c.decode('utf-8').split('/')[-2]  # 從路徑字串擷取影片 ID（倒數第二層資料夾名）。
        if c not in video2labels:  # 若該影片尚未建立儲存空間。
          video2labels[c] = []  # 初始化該影片的真實標籤清單。
          video2preds[c] = []  # 初始化該影片的預測標籤清單。
        video2labels[c].append(l)  # 將目前 clip 的真實標籤加入對應影片。
        video2preds[c].append(p)  # 將目前 clip 的預測標籤加入對應影片。

  # anyway calculate frame-level metrics  # 註解：無論最終聚合方式，都先計算 frame 層級指標供後續使用。
  all_labels = np.asarray(all_labels)  # 將總真實標籤清單轉為 NumPy 陣列。
  all_preds = np.asarray(all_preds)  # 將總預測標籤清單轉為 NumPy 陣列。
  frame_mets = classification_report(all_labels, all_preds, output_dict=True)  # 產生 frame 層級分類報告（字典格式）。

  if agg == 'frame':  # 若指定回傳 frame 層級結果。
    all_labels = np.asarray(all_labels)  # 確保真實標籤為 NumPy 陣列。
    all_preds = np.asarray(all_preds)  # 確保預測標籤為 NumPy 陣列。

    return {  # 回傳 frame 層級主要評估指標。
        'acc': np.round(  # 計算並四捨五入 accuracy（百分比）。
            np.sum(all_labels == all_preds) * 100 / len(all_labels), 2  # 以正確數/總數計算準確率。
        ),  # 結束 acc 計算。
        'f1': np.round(score(all_labels, all_preds)[-2].mean() * 100, 2),  # 取各類別 f1 平均後轉成百分比並四捨五入。
    }  # 結束 frame 模式回傳。

  elif agg == 'video':  # 若指定回傳 video 層級結果。
    accs = []  # 建立清單：存每部影片的準確率。
    scores = []  # 建立清單：存每部影片的 precision/recall/f1/support 平均值。
    for sub_labels, sub_preds in zip(  # 逐部影片配對真實標籤序列與預測序列。
        video2labels.values(), video2preds.values()  # 取出所有影片的標籤與預測集合。
    ):  # 結束影片逐一迭代定義。
      sub_labels = np.asarray(sub_labels)  # 該影片真實標籤轉為 NumPy 陣列。
      sub_preds = np.asarray(sub_preds)  # 該影片預測標籤轉為 NumPy 陣列。

      # compute acc and append  # 註解：先計算單一影片準確率並加入清單。
      vid_acc = np.sum(sub_labels == sub_preds) * 100 / len(sub_labels)  # 計算該影片準確率（百分比）。
      accs.append(vid_acc)  # 將該影片準確率加入總清單。

      # compute F1  # 註解：計算單一影片 precision/recall/f1 等指標。
      vid_score = score(sub_labels, sub_preds)  # 取得該影片各類別 precision/recall/f1/support。
      mean = np.mean(np.vstack(vid_score).T, axis=0)  # 對各類別做平均，得到單一影片整體指標。
      mean[:-1] *= 100  # 將 precision/recall/f1 轉為百分比（support 不轉）。
      scores.append(mean)  # 將該影片指標加入總清單。

    # summarize  # 註解：彙整所有影片的指標。
    overall_acc = np.around(np.mean(np.stack(accs)), 2)  # 計算所有影片平均準確率並四捨五入。
    overall_f1 = np.mean(np.stack(scores), axis=0)  # 計算所有影片平均 precision/recall/f1/support。
    overall_f1 = np.around(overall_f1, 2)  # 對上述平均結果做四捨五入。

    return {  # 回傳 video 與 frame 層級摘要結果。
        'video_acc': overall_acc,  # 影片層級平均準確率。
        'video_precision': overall_f1[0],  # 影片層級平均 precision。
        'video_recall': overall_f1[1],  # 影片層級平均 recall。
        'video_f1': overall_f1[2],  # 影片層級平均 f1。

        'frame_acc': np.around(  # frame 層級 accuracy（百分比）。
            frame_mets['accuracy'] * 100, 2),  # 從分類報告取 accuracy 後轉百分比並四捨五入。
        'frame_macro_f1': np.around(  # frame 層級 macro-f1。
            frame_mets['macro avg']['f1-score'] * 100, 2),  # 從分類報告取 macro avg f1-score。
        'frame_micro_f1': np.around(  # frame 層級 weighted/micro 風格 f1（原程式使用 weighted avg）。
            frame_mets['weighted avg']['f1-score'] * 100, 2),  # 從分類報告取 weighted avg f1-score。
        }  # 結束 video 模式回傳。


def calc_map(model, ds, agg='all', verbose=0):  # 定義 mAP 計算函式；可回傳整體或分類別結果。

  all_labels = np.empty(())  # 先建立空陣列作為標籤容器（稍後會被第一批資料覆蓋）。
  all_preds = np.empty(())  # 先建立空陣列作為預測容器（稍後會被第一批資料覆蓋）。
  for i, batch in enumerate(ds):  # 逐批走訪資料集並取得批次索引。
    inputs, labels = batch  # 拆出輸入與標籤（此任務不含 clip 路徑）。
    preds = model.predict(inputs, verbose=verbose)  # 對該批輸入做推論。
    if i == 0:  # 若為第一批，直接初始化總容器。
      all_labels = labels.numpy()  # 以第一批真實標籤初始化。
      all_preds = preds  # 以第一批預測結果初始化。
    else:  # 若非第一批，則與既有資料串接。
      all_labels = np.concatenate((all_labels, labels.numpy()), 0)  # 在第 0 維串接真實標籤。
      all_preds = np.concatenate((all_preds, preds), 0)  # 在第 0 維串接預測分數。
  try:  # 嘗試計算 mAP，避免評估時因資料異常中斷。
    mean = [mean_ap(all_labels, all_preds) * 100]  # 計算整體 mAP 並轉百分比（包成 list 方便統一格式）。
    std = [0.00]  # 此實作未估標準差，固定回傳 0。
    if agg == 'class':  # 若要求分類別結果。
      mean = mean_ap(all_labels, all_preds, mean=False) * 100  # 改為取得每一類別的 AP。
      std = [0.00] * 7  # 分類別模式下回傳對應長度的 0 標準差陣列（此處固定 7 類）。
  except:  # 若計算 AP 發生例外（例如標籤分布不合法）。
    mean = [-1] * 7 if agg == 'class' else [-1.00]  # 失敗時以 -1 當作指標無效值。
    std = [0.00] * 7 if agg == 'class' else [0.00]  # 失敗時標準差維持 0。
  mean = [np.round(i, 2) for i in mean]  # 將 mean 結果逐一四捨五入到小數點後兩位。
  std = [np.round(i, 2) for i in std]  # 將 std 結果逐一四捨五入到小數點後兩位。
  if len(mean) == 1:  # 若只有單一整體指標。
    return {'mean': mean[0], 'std': std[0]}  # 回傳標量格式的 mean/std。
  return {'mean': mean, 'std': std}  # 否則回傳向量格式的分類別 mean/std。


def mean_ap(labels, predictions, mean=True):  # 定義 AP 計算輔助函式。
  metrics = np.array(average_precision_score(labels, predictions, average=None))  # 計算各類別 AP 並轉為 NumPy 陣列。
  if mean:  # 若需要整體平均 AP。
    metrics = np.sum([x for x in metrics]) / len(metrics)  # 以各類 AP 的算術平均作為 mAP。
  return metrics  # 回傳單一 mAP 或各類別 AP。

def end_of_training_evaluation(  # 定義訓練結束時的整體評估函式。
    model, validation_ds, test_ds, label_key, exp_dir, epoch):  # 輸入模型、驗證/測試集、任務類型、輸出目錄與 epoch。
  """Run aggregative evaluation over the trained model."""  # 函式說明：對訓練完成模型執行聚合評估。

  if label_key == 'tool':  # 若任務標籤為 tool（多標籤 mAP 任務）。
    validation_map = calc_map(model, validation_ds)  # 計算驗證集 mAP。
    test_map = calc_map(model, test_ds)  # 計算測試集 mAP。
    mets = {'val_map': validation_map['mean'], 'test_map': test_map['mean']}  # 整理成統一字典輸出。

  elif label_key == 'segment':  # 若任務標籤為 segment（分類 F1 任務）。
    mets = {}  # 建立空字典存放各項指標。
    for k, v in calc_f1(model, validation_ds, agg='video').items():  # 計算驗證集 video 聚合指標並逐項走訪。
      mets[f'val_{k}'] = v  # 將驗證集指標加上 val_ 前綴存入結果。
    for k, v in calc_f1(model, test_ds, agg='video').items():  # 計算測試集 video 聚合指標並逐項走訪。
      mets[f'test_{k}'] = v  # 將測試集指標加上 test_ 前綴存入結果。

  else:  # 若 label_key 非預期值。
    raise ValueError  # 拋出錯誤提醒呼叫端輸入不合法。

  file_writer = tf.summary.create_file_writer(exp_dir + '/metrics')  # 建立 TensorBoard writer，輸出到 metrics 子目錄。
  file_writer.set_as_default()  # 設定此 writer 為預設 writer。
  for k, v in mets.items():  # 逐一走訪所有評估指標。
    tf.summary.scalar(k, data=v, step=epoch)  # 以 scalar 形式寫入 TensorBoard，step 使用當前 epoch。
  return mets  # 回傳最終評估指標字典。

down_stream/config.py

In [6]:
"""Base config class."""

import os

# DRIVE_DIR = '/content/gdrive/MyDrive/PMBM/論文程式/MSM'
# par_dir = os.path.realpath(__file__+ '/../../')
par_dir= DATA_ROOT


class Config:
  """Base config class."""

  def __init__(self, **kwargs):
    for k, v in kwargs.items():
      setattr(self, k, v)

  exp_dir = os.path.join(par_dir, 'exps', 'tmp')  # 2026/3/6 這裡面要放什麼?

  # dataset
  dataset_name = 'cholec80'
  # data_root = os.path.join(par_dir, 'cholec80', 'cholec80')
  data_root = os.path.join(par_dir, 'cholec80', 'cholec80_extracted')  # 組合預設資料集根目錄。

  train_transformation = 'randaug'
  label_key = 'segment'

  is_linear_evaluation = False
  model = 'resnet50'
  saved_model_dir = os.path.join(par_dir, 'checkpoints', 'pretrained_model_dir') # 2026/3/6 需要確認modle的用途
  task_type = 'multi_class'
  monitor_metric = 'val_macro_f1'
  input_dim = {'vits': 384, 'vitb': 768, 'vitl': 1024, 'resnet50': 2048}[model]
  num_classes = 7

  # optimization
  use_class_weight = False
  optimize_name = 'adamw'
  learning_rate = 1e-4
  momentum = 0.9
  weight_decay = 1e-5

  num_epochs = 5
  batch_size = 256
  validation_freq = 1
  callbacks_names = ['checkpoint', 'reduce_lr_plateau', 'early_stopping']
  manually_load_best_checkpoint = True

down_stream/experiment.py

In [7]:
"""A compact module for running down-stream experiments."""  # 模組說明：用於執行下游任務實驗。

import sys  # 匯入 sys，以便調整 Python 模組搜尋路徑。
import os  # 匯入 os，供檔案路徑與資料夾操作使用。
# sys.path.append(os.path.realpath(__file__ + '/../../'))  # 將專案根目錄加入路徑，讓跨資料夾模組可被匯入。 在colab 應該用不到，要處一存檔的位置

import tensorflow as tf  # 匯入 TensorFlow，建立與訓練模型。

# from data import cholec80_images  # 匯入資料集工具，建立 Cholec80 影像資料集。
# from train import eval_lib  # 匯入評估函式庫，做訓練後驗證/測試評估。
# from train import train_lib  # 匯入訓練函式庫，提供模型、優化器、callbacks 等工具。


def verbose_print(msg, verbose=True, is_title=False):  # 定義可控輸出的列印函式。
  if verbose:  # 若 verbose 為 True，才印出一般訊息。
    print(msg)  # 輸出傳入的訊息內容。
  if is_title:  # 若此訊息被標記為標題。
      print('#' * 40)  # 額外印出分隔線，方便閱讀日誌區段。


def run_experiment(config, verbose=True):  # 定義主流程：根據設定執行完整實驗。
  """Stand-alone function for running an experiment."""  # 函式說明：可獨立呼叫以執行一次實驗。

  verbose_print('Config:', verbose)  # 先印出「Config」標題。
  attr_names = [i for i in dir(config) if not i.startswith('__')]  # 取得 config 物件中可見且非內建屬性名稱。
  for a in attr_names:  # 逐一走訪每個設定欄位。
    verbose_print('{} = {}'.format(a, getattr(config, a, None)), verbose)  # 印出欄位名稱與對應值。
  print('\n\n')  # 輸出空行，分隔設定區與後續流程。

  if not os.path.exists(config.exp_dir):  # 若實驗輸出目錄不存在。
    os.makedirs(config.exp_dir)  # 建立實驗輸出目錄。

  ##############################################################################  # 區塊分隔：資料集與模型建立。
  # Datasets & Model  # 區塊標題：資料集與模型。
  ##############################################################################  # 區塊分隔線。
  verbose_print('Create datasets and model', verbose, True)  # 印出建立資料與模型的標題訊息。
  datasets = get_cholec80_images_datasets(  # 呼叫資料函式，取得 train/validation/test 資料集。
      data_root=config.data_root,  # 指定資料根目錄。
      batch_size=config.batch_size,  # 指定批次大小。
      train_transformation=config.train_transformation,  # 指定訓練資料增強/轉換設定。
  )  # 結束資料集建立呼叫。


  if config.is_linear_evaluation:  # 若設定為線性評估模式（凍結 backbone，只訓練線性層）。
    model = get_linear_model(  # 建立線性分類模型。
        input_dim=config.input_dim, output_dim=config.num_classes  # 設定輸入特徵維度與輸出類別數。
    )  # 結束線性模型建立。
  elif config.model == 'resnet50':  # 否則若模型類型指定為 ResNet50V2。
    input_tensor = tf.keras.Input(shape=(224, 224, 3,))  # 建立模型輸入張量（224x224 RGB）。
    backbone = tf.keras.applications.resnet_v2.ResNet50V2(  # 建立 ResNet50V2 主幹網路。
        include_top=False,  # 不使用 ImageNet 原本頂層分類器。
        weights='imagenet',  # 載入 ImageNet 預訓練權重。
        input_tensor=input_tensor)  # 將自訂輸入張量接入 backbone。
    out = tf.keras.layers.GlobalAveragePooling2D()(backbone.output)  # 對特徵圖做全域平均池化。
    out = tf.keras.layers.Dense(config.num_classes)(out)  # 接上分類 Dense 層，輸出類別 logits。
    model = tf.keras.Model(input_tensor, out, name='Model')  # 將輸入與輸出封裝為完整 Keras 模型。
  elif 'vit' in config.model:  # 否則若模型名稱含 vit（Vision Transformer）。
    backbone = tf.saved_model.load(config.saved_model_dir)  # 從 saved_model_dir 載入預訓練 ViT backbone。
    model = LinearFineTuneModel(backbone, config.num_classes)  # 包裝為可微調的線性分類模型。
  else:  # 若上述模型條件皆不符合。
    raise ValueError('Invalid model name: {}'.format(config.model))  # 拋出錯誤，提示無效模型名稱。

  model.compile(  # 編譯模型，設定優化器、損失函數、評估指標。
      optimizer=get_optimizer(  # 依 config 建立優化器。
          config.optimize_name,  # 優化器名稱（如 SGD/Adam）。
          config.learning_rate,  # 學習率。
          config.momentum,  # 動量參數（若優化器支援）。
          config.weight_decay,  # 權重衰減係數。
      ),  # 結束優化器設定。
      loss=get_loss(config.task_type),  # 依任務型態取得對應 loss。
      metrics=get_metrics(config.task_type, config.num_classes),  # 依任務型態與類別數設定 metrics。
  )  # 結束 compile。

  ##############################################################################  # 區塊分隔：開始訓練。
  # Train  # 區塊標題：訓練。
  ##############################################################################  # 區塊分隔線。
  verbose_print('Begin training', verbose, True)  # 印出訓練開始標題。
  history = model.fit(  # 執行模型訓練，並回傳歷史紀錄。
      datasets['train'],  # 使用訓練資料集。
      batch_size=config.batch_size,  # 訓練批次大小。
      epochs=config.num_epochs,  # 訓練 epoch 數。
      validation_data=datasets['validation'],  # 設定驗證資料集。
      class_weight=(_CHOLEC80_PHASES_WEIGHTS  # 若啟用，使用類別權重平衡不平衡資料。
                    if config.use_class_weight else None),  # 否則不使用 class weight。
      callbacks=get_callbacks(  # 建立並設定訓練 callbacks。
          callbacks_names=config.callbacks_names,  # callback 名稱清單。
          exp_dir=config.exp_dir,  # 實驗輸出路徑（存 checkpoint/log）。
          monitor_metric=config.monitor_metric,  # callback 監控指標。
          learning_rate=config.learning_rate,  # callback 可能需要的初始學習率。
      ),  # 結束 callbacks 設定。
      validation_freq=config.validation_freq,  # 每幾個 epoch 做一次驗證。
  )  # 結束 fit 呼叫。

  #############################################################################  # 區塊分隔：訓練結束後評估。
  # End of train evaluation  # 區塊標題：訓練結束評估。
  #############################################################################  # 區塊分隔線。
  if config.manually_load_best_checkpoint:  # 若設定需要手動載入最佳/最新 checkpoint。
    checkpoints = os.listdir(os.path.join(config.exp_dir, 'checkpoints') + '/*')  # 嘗試讀取 checkpoint 路徑下的檔案清單。
    if checkpoints:  # 若有找到 checkpoint。
      latest = checkpoints[-1]  # 取清單最後一個作為最新 checkpoint。
      print(f'Load latest checkpoint: {latest}')  # 印出即將載入的 checkpoint 路徑。
      model.load_weights(latest)  # 載入 checkpoint 權重。
    else:  # 若沒有 checkpoint 可載入。
      print('Haven\'t loaded a saved checkpoint')  # 提示未載入任何已儲存權重。

  # For the special case of phases, re-extract the dataset with the 'with_image_path'  # 說明：phase 任務需重新取資料，附帶影像路徑。
  # attribute for calculating video-level metrics  # 原因：後續要計算影片層級指標。
  verbose_print('Start end of train evaluation', verbose, True)  # 印出進入訓練後評估流程的標題。
  datasets = get_cholec80_images_datasets(  # 重新建立資料集（含 image path）。
      data_root=config.data_root,  # 指定資料根目錄。
      batch_size=config.batch_size,  # 指定批次大小。
      train_transformation=config.train_transformation,  # 保持相同訓練轉換設定。
      with_image_path=True,  # 額外回傳影像路徑，供影片層級評估使用。
  )  # 結束資料集重建。

  mets = end_of_training_evaluation(  # 執行訓練結束評估並取得指標。
      model,  # 傳入已訓練模型。
      datasets['validation'],  # 傳入驗證集。
      datasets['test'],  # 傳入測試集。
      label_key=config.label_key,  # 指定標籤欄位名稱。
      exp_dir=config.exp_dir,  # 指定輸出目錄（儲存評估結果）。
      epoch=config.num_epochs)  # 指定評估所屬 epoch（通常為最終 epoch）。
  return mets, history  # 回傳評估指標與訓練歷史。

down_stream/main.py

In [ ]:
"""主程式：用於線性評估（linear evaluation）或微調（fine-tuning）。"""

# 匯入 argparse，用來解析命令列參數。
import argparse
# 匯入本專案的設定模組，提供 Config 類別。
# import config  #在colab 內已經執行了
# 匯入實驗執行模組，負責啟動訓練/評估流程。
# import experiment #在colab 內已經執行了


# 建立命令列參數解析器物件。
parser = argparse.ArgumentParser()

# 以下皆為可選參數，主要用於超參數掃描（parameter sweep）。
# 新增 --model 參數，讓使用者指定模型名稱。
parser.add_argument(
  '--model',
  type=str,
  required=False)

# 新增 --optimizer 參數，讓使用者指定最佳化器類型。
parser.add_argument(
  '--optimizer',
  type=str,
  required=False)

# 新增 --learning_rate 參數，讓使用者覆寫學習率。
parser.add_argument(
  '--learning_rate',
  type=float,
  required=False)

# 新增 --weight_decay 參數，讓使用者覆寫權重衰減值。
parser.add_argument(
  '--weight_decay',
  type=float,
  required=False)


# 定義主函式，接收解析後的命令列參數。
def main(args):
    # 建立預設設定物件（包含專案預設超參數）。
    conf = Config()
    # 逐一巡覽命令列參數（名稱 k、值 v）。
    for k, v in vars(args).items():
        # 只有當參數有提供有效值時才覆寫設定。
        if v:
            # 將命令列提供的值寫回設定物件對應欄位。
            setattr(conf, k, v)
    # 依照最終設定執行實驗流程（回傳值此處不使用）。
    _ = run_experiment(conf)

# 確保只有直接執行此檔案時才會啟動主程式。
if __name__ == '__main__':
  # 解析命令列輸入為 args 物件。
  # 修正：在 Colab 中，sys.argv 包含 kernel 啟動參數 (-f ...)，需傳入 args=[] 忽略之。
  args = parser.parse_args(args=[])
  # 呼叫主函式啟動流程。
  main(args)

Config:
batch_size = 256
callbacks_names = ['checkpoint', 'reduce_lr_plateau', 'early_stopping']
data_root = /content/gdrive/MyDrive/PMBM/論文程式/MSM/cholec80/cholec80_extracted
dataset_name = cholec80
exp_dir = /content/gdrive/MyDrive/PMBM/論文程式/MSM/exps/tmp
input_dim = 2048
is_linear_evaluation = False
label_key = segment
learning_rate = 0.0001
manually_load_best_checkpoint = True
model = resnet50
momentum = 0.9
monitor_metric = val_macro_f1
num_classes = 7
num_epochs = 5
optimize_name = adamw
saved_model_dir = /content/gdrive/MyDrive/PMBM/論文程式/MSM/checkpoints/pretrained_model_dir
task_type = multi_class
train_transformation = randaug
use_class_weight = False
validation_freq = 1
weight_decay = 1e-05



Create datasets and model
########################################
94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Begin training
########################################
Epoch 1/5
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 28s/step - loss: 1.6763 - macro_f1: 0.1125 - micro_f1: 0.3433 - sparse_cat